# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines run over the **same chunked corpus** and are compared:

- **Pipeline 1 — Baseline:** dense bi-encoder retrieval. Embed the query, pull the top 10 chunks by cosine similarity, hand the top 3 to the LLM.
- **Pipeline 2 — Re-ranking:** take the same top 10, re-score every `(query, chunk)` pair with a cross-encoder, re-sort, and hand the new top 3 to the LLM.

The corpus is the Catan base-game rulebook (rules + almanac) — dense with exceptions, penalties and role limits, which makes retrieval mistakes easy to see.

## 1. Setup & Environment

Models used:
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2`
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Answer generation: `Qwen/Qwen2.5-1.5B-Instruct`, run locally with Hugging Face `transformers`

All three models download from Hugging Face the first time they are used and then run on the local CPU. No API key or network service is required.

In [1]:
import os, re, time, textwrap, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pypdf
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Retrieval settings
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10   # candidates pulled by the baseline
TOP_N = 3    # chunks actually sent to the LLM

# Chunking settings
CHUNK_CHARS = 700
CHUNK_OVERLAP = 120

# Answer generation (local Hugging Face model)
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
print("Answer model (local):", LOCAL_MODEL)

C:\Users\PARAM MODI\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Answer model (local): Qwen/Qwen2.5-1.5B-Instruct


## 2. Dataset

A single complex PDF is the knowledge base. The notebook downloads it on first run and keeps it in `data/`.

Each page is extracted separately so every chunk can be traced back to a page number.

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "catan_rules.pdf"
PDF_URL = "https://world-of-board-games.com.sg/docs/Catan-5ed.pdf"

if not PDF_PATH.exists():
    print("Downloading corpus ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print("PDF size (KB):", PDF_PATH.stat().st_size // 1024)


def load_pages(path):
    reader = pypdf.PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages


pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(pages[1]["text"][:400])

PDF size (KB): 3575


Loaded 16 non-empty pages
22
Dear Settlers,
To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system, which consists 
of 3 parts. First, if you do not know how to play Catan, please read the Game Overview on page 16 (the back cover). 
Next, read the Game Rules on pages 2-6 (red borders) and start to play the game. If you have questions during the 
game, consult the Catan Almanac 


## 3. Chunking

The pages are split into overlapping character windows. Paragraph boundaries are respected where possible so a rule and its exception tend to stay together, and a small overlap keeps context from being cut mid-sentence. Each chunk records its source page.

In [3]:
def chunk_pages(pages, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    chunks = []
    for item in pages:
        page_no, text = item["page"], item["text"]
        paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
        buf = ""
        for para in paragraphs:
            if len(buf) + len(para) + 1 <= size:
                buf = (buf + " " + para).strip()
            else:
                if buf:
                    chunks.append({"page": page_no, "text": buf})
                if len(para) > size:
                    # a single long paragraph: slide a window across it
                    for start in range(0, len(para), size - overlap):
                        chunks.append({"page": page_no, "text": para[start:start + size]})
                    buf = ""
                else:
                    tail = buf[-overlap:] if buf else ""
                    buf = (tail + " " + para).strip()
        if buf:
            chunks.append({"page": page_no, "text": buf})
    return chunks


chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(t) for t in chunk_texts) // len(chunk_texts)} chars")
print("\nExample chunk:\n", chunk_texts[5])

Total chunks: 100
Average chunk length: 635 chars

Example chunk:
 cut sheets, always push the tiles through from the front, “cut” side (pushing from the back may cause the tiles to rip). Lay out the map as specified in Illustration A (or on the back of this booklet). First, assemble the frame as shown. Second, create Catan by placing the 19 terrain hexes on the table—again as shown. Third, place the circular number tokens on top of the designated terrain hexes. Finally, place your settlements and roads. Africa is spring and we walk five paths toward the sun


## 4. Pipeline 1 — Baseline Retrieval (Bi-Encoder)

The bi-encoder embeds each chunk independently into a vector. Vectors are L2-normalised and stored in a FAISS inner-product index, so an inner-product search is equivalent to cosine similarity. For a query we embed it the same way and pull the `TOP_K` nearest chunks.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

chunk_embeddings = bi_encoder.encode(
    chunk_texts, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks, dimension {dim}")


def baseline_retrieve(query, k=TOP_K):
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"id": int(i), "score": float(s)} for i, s in zip(ids[0], scores[0])]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2150.60it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:  25%|██▌       | 1/4 [00:01<00:05,  1.93s/it]

Batches:  50%|█████     | 2/4 [00:03<00:03,  1.91s/it]

Batches:  75%|███████▌  | 3/4 [00:05<00:01,  1.83s/it]

Batches: 100%|██████████| 4/4 [00:05<00:00,  1.15s/it]

Batches: 100%|██████████| 4/4 [00:05<00:00,  1.42s/it]

Indexed 100 chunks, dimension 384


## 5. Pipeline 2 — Re-Ranking (Cross-Encoder)

The cross-encoder takes the `TOP_K` baseline candidates and reads each `(query, chunk)` pair **together** in one pass, so its score reflects direct interaction between the query and the passage rather than two separate embeddings. We re-sort the 10 candidates by this score and keep the new top 3.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def rerank(query, candidates):
    ids = [c["id"] for c in candidates]
    pairs = [(query, chunk_texts[i]) for i in ids]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(
        ({"id": i, "score": float(s)} for i, s in zip(ids, scores)),
        key=lambda x: x["score"], reverse=True,
    )
    return ranked

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1650.83it/s]

## 6. Prompt & Answer Generation

Both pipelines use the **identical prompt structure** — only the three retrieved chunks differ. The model is told to answer strictly from the passages and to say so when the answer is absent, which is what exposes a weak retrieval: if the critical chunk is missing, the answer is wrong or incomplete.

The model is loaded once (lazily) and reused. Greedy decoding (`do_sample=False`) keeps the output deterministic, so the two pipelines are compared on their retrieved context, not on sampling noise.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

_tokenizer = None
_model = None


def load_llm():
    global _tokenizer, _model
    if _model is None:
        _tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
        _model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float32)
        _model.eval()
    return _tokenizer, _model


SYSTEM_PROMPT = (
    "You answer board-game rules questions using only the provided context passages. "
    "Quote the specific rule and cite the passage numbers you used. "
    "If the answer is not contained in the passages, say that you cannot find it."
)


def format_context(ids):
    blocks = []
    for rank, i in enumerate(ids, start=1):
        blocks.append(f"[{rank}] (page {chunks[i]['page']}) {chunk_texts[i]}")
    return "\n\n".join(blocks)


def build_user_prompt(query, ids):
    return f"Context passages:\n{format_context(ids)}\n\nQuestion: {query}\n\nAnswer:"


def generate_answer(query, ids, max_new_tokens=256):
    tokenizer, model = load_llm()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(query, ids)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()

## 7. Presentation Helpers & End-to-End Check

These helpers render each query's retrieval as a single table — every candidate with its baseline rank, its re-ranked rank and both scores — and print the two answers as formatted text. Colour coding: **green** rows are the top 3 actually sent to the LLM, **amber** rows are chunks the cross-encoder lifted from ranks 4–10 into the top 3.

Below them is a sanity run on an easy question, where the baseline is already expected to do well.

In [7]:
from IPython.display import display, Markdown

SENT_TO_LLM = "background-color: #d1e7dd; color: #0f5132"   # kept in the top 3
PROMOTED = "background-color: #fff3cd; color: #664d03"      # lifted from 4-10 into the top 3


def preview(i, n=70):
    return " ".join(chunk_texts[i].split())[:n]


def run_both(query):
    base = baseline_retrieve(query, TOP_K)
    base_ids = [c["id"] for c in base]
    reranked = rerank(query, base)
    rr_ids = [c["id"] for c in reranked]
    return base, base_ids, reranked, rr_ids


def ranking_table(base, reranked):
    base_ids = [c["id"] for c in base]
    rr_ids = [c["id"] for c in reranked]
    bscore = {c["id"]: c["score"] for c in base}
    rscore = {c["id"]: c["score"] for c in reranked}
    rows = []
    for i in base_ids:
        br, rr = base_ids.index(i) + 1, rr_ids.index(i) + 1
        rows.append({"chunk": i, "page": chunks[i]["page"],
                     "baseline #": br, "baseline score": bscore[i],
                     "rerank #": rr, "rerank score": rscore[i],
                     "promoted": "yes" if (br > TOP_N and rr <= TOP_N) else "",
                     "preview": preview(i)})
    return pd.DataFrame(rows).sort_values("rerank #").reset_index(drop=True)


def style_ranking(df, caption=""):
    def row_style(r):
        if r["promoted"] == "yes":
            return [PROMOTED] * len(r)
        if r["rerank #"] <= TOP_N:
            return [SENT_TO_LLM] * len(r)
        return [""] * len(r)
    s = (df.style.apply(row_style, axis=1).hide(axis="index")
         .format({"baseline score": "{:.3f}", "rerank score": "{:.3f}"}))
    return s.set_caption(caption) if caption else s


def show_answers(answer_1, answer_2):
    display(Markdown("**Answer 1 — baseline top 3**"))
    display(Markdown(answer_1))
    display(Markdown("**Answer 2 — re-ranked top 3**"))
    display(Markdown(answer_2))

In [8]:
demo_query = "What does a player need to do to win the game?"
display(Markdown(f"**Demo query:** {demo_query}"))

base, base_ids, reranked, rr_ids = run_both(demo_query)
display(style_ranking(ranking_table(base, reranked),
                      "Baseline vs. re-ranked order (green = sent to the LLM)"))

show_answers(generate_answer(demo_query, base_ids[:TOP_N]),
             generate_answer(demo_query, rr_ids[:TOP_N]))

**Demo query:** What does a player need to do to win the game?

chunk,page,baseline #,baseline score,rerank #,rerank score,promoted,preview
81,14,1,0.460,1,3.621,,"end of the game. Place them face down in front of you. Of course, if y"
82,14,9,0.338,2,3.192,yes,"ents, each player begins the game with 2 victory points. Therefore, yo"
25,5,3,0.396,3,0.697,,Victory Point Cards (Orange Frame) Y You must keep victory point cards
39,7,10,0.336,4,-1.203,,"m victory. Example: Siobhán has 2 settlements (2 points), the Longest"
38,7,6,0.371,5,-1.212,,"ve me 1 brick for 1 ore?” Beth answers, “If you give me 3 ore, I’ll gi"
45,8,8,0.347,6,-2.459,,"you get nothing! (However, you can always ask players about the number"
24,5,5,0.377,7,-2.598,,"bber” above and follow steps 1 and 2. Once played, knight cards remain"
12,4,2,0.436,8,-2.875,,"ess you’re using the Starting Set-Up for Experienced Players, the olde"
9,3,7,0.351,9,-3.394,,"let. Next, read the Game Rules and start to play. And finally, if you"
26,5,4,0.391,10,-7.839,,r (including you) has 10 points on their turn. © 2015 Catan GmbH © 201


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   6%|▌         | 21/338 [00:00<00:03, 100.85it/s]

Loading weights:   9%|▉         | 32/338 [00:00<00:04, 68.27it/s] 

...

Loading weights: 100%|██████████| 338/338 [00:08<00:00, 38.39it/s]

**Answer 1 — baseline top 3**

A player needs to reach 10 victory points on their turn to win the game. This is stated directly in the passage: "The first player to reach (be at) 10 victory points (VPs) on their turn wins the game."

**Answer 2 — re-ranked top 3**

A player needs to reach 10 victory points on their turn to win the game.

## 8. Failure Mode Analysis (Stress Testing)

Each query below is chosen so the critical chunk sits **outside** the baseline top 3 (buried at ranks 4–10, or missed on wording) but the cross-encoder promotes it into the top 3. Categories covered: keyword trap / distractor, exception & corner case, and negation / constraint.

In [9]:
FAILURE_QUERIES = [
    {
        "category": "Keyword trap / distractor",
        "query": "What happens when a seven is rolled?",
        "note": "The baseline is pulled toward passages that only share wording - the number-token "
                "text that says 'there is no 7', and the table of contents - and misses the actual "
                "robber-and-discard procedure.",
    },
    {
        "category": "Exception & corner case",
        "query": "Can I build a settlement right next to another settlement?",
        "note": "The generic 'how to build a settlement' text ranks high; the Distance Rule "
                "restriction that actually answers the question sits further down.",
    },
    {
        "category": "Negation / constraint",
        "query": "Can I give away resource cards to another player as a gift?",
        "note": "The governing rule is a prohibition ('you cannot give away cards'); the bi-encoder "
                "blurs the negative and surfaces generic trading text instead.",
    },
    {
        "category": "Buried specific (ranks 4-10)",
        "query": "How many points do I need to win?",
        "note": "The exact victory-point total appears in one worked example, buried below several "
                "passages that mention points in general.",
    },
]


def analyse(query, note="", category=""):
    base, base_ids, reranked, rr_ids = run_both(query)
    display(Markdown(f"### {category}\n**Q: {query}**  \n_{note}_"))
    display(style_ranking(ranking_table(base, reranked),
                          "green = sent to the LLM, amber = promoted into the top 3"))

    promoted = [i for i in rr_ids[:TOP_N] if i in base_ids[TOP_N:]]
    records, lines = [], []
    for pid in promoted:
        before, after = base_ids.index(pid) + 1, rr_ids.index(pid) + 1
        records.append({"query": query, "category": category, "chunk": pid,
                        "page": chunks[pid]["page"],
                        "baseline_rank": before, "rerank_rank": after})
        lines.append(f"- **chunk {pid}** (page {chunks[pid]['page']}): "
                     f"baseline #{before} → re-rank #{after}  \n  _{preview(pid, 150)}_")
    if lines:
        display(Markdown("**Promoted into the top 3**\n\n" + "\n".join(lines)))

    show_answers(generate_answer(query, base_ids[:TOP_N]),
                 generate_answer(query, rr_ids[:TOP_N]))
    return records


rank_shift_records = []
for item in FAILURE_QUERIES:
    rank_shift_records += analyse(item["query"], item["note"], item["category"])

### Keyword trap / distractor
**Q: What happens when a seven is rolled?**  
_The baseline is pulled toward passages that only share wording - the number-token text that says 'there is no 7', and the table of contents - and misses the actual robber-and-discard procedure._

chunk,page,baseline #,baseline score,rerank #,rerank score,promoted,preview
23,5,6,0.297,1,-0.482,yes,Important: If the production number for the hex containing the robber
63,11,8,0.293,2,-1.440,yes,and/or cities adjacent to the target terrain hex receive no resources
64,11,4,0.365,3,-1.529,yes,"er with more than 7 resource cards (i.e., 8 or more) must choose and d"
53,10,2,0.394,4,-1.684,,1010 NN numBer toKens The 18 number tokens are marked with the numeral
57,10,1,0.433,5,-2.857,,is rolled (exception: see Robber Y). resour Ce produ Ction On your tur
94,16,10,0.265,6,-5.608,,"dering these terrain hexes. In the illustration, the red settlement [A"
66,11,5,0.301,7,-5.857,,r 2 hexes meet the frame). You share in all of the resource production
9,3,9,0.269,8,-8.422,,"let. Next, read the Game Rules and start to play. And finally, if you"
69,12,7,0.296,9,-10.812,,tarting player and begins. The starting player places a settlement on
88,15,3,0.368,10,-10.906,,er T okens . . . . . . . . . . . . . . . . . . . . . . 10 Paths . . .


**Promoted into the top 3**

- **chunk 23** (page 5): baseline #6 → re-rank #1  
  _Important: If the production number for the hex containing the robber is rolled, the owners of adjacent settlements and cities do not receive resource_
- **chunk 63** (page 11): baseline #8 → re-rank #2  
  _and/or cities adjacent to the target terrain hex receive no resources from this hex as long as the robber is in the hex. “A” and “B” do not receive a _
- **chunk 64** (page 11): baseline #4 → re-rank #3  
  _er with more than 7 resource cards (i.e., 8 or more) must choose and discard half of them. Return discards to the supply stacks. If you hold an odd nu_

**Answer 1 — baseline top 3**

When a "7" is rolled, the robber token is activated.

...

**Promoted into the top 3**

- **chunk 39** (page 7): baseline #4 → re-rank #1  
  _m victory. Example: Siobhán has 2 settlements (2 points), the Longest Road special card (2 points), 2 cities (4 points), and 2 victory point cards (2 _

**Answer 1 — baseline top 3**

To win the game, you need to reach 10 victory points on your turn.

**Answer 2 — re-ranked top 3**

To win the game, you need 10 victory points.

## 9. Rank-Shift Summary

The chunks that the cross-encoder lifted from ranks 4–10 into the top 3. These are exactly the passages the baseline would have withheld from the LLM.

In [10]:
rank_shift_df = pd.DataFrame(rank_shift_records,
                             columns=["query", "category", "chunk", "page",
                                      "baseline_rank", "rerank_rank"])
rank_shift_df["shift"] = rank_shift_df["baseline_rank"] - rank_shift_df["rerank_rank"]
(rank_shift_df.style.hide(axis="index")
    .bar(subset=["shift"], color="#9ec5fe", align="left")
    .set_caption("Chunks promoted from ranks 4-10 into the top 3 (shift = ranks gained)"))

query,category,chunk,page,baseline_rank,rerank_rank,shift
What happens when a seven is rolled?,Keyword trap / distractor,23,5,6,1,5
What happens when a seven is rolled?,Keyword trap / distractor,63,11,8,2,6
What happens when a seven is rolled?,Keyword trap / distractor,64,11,4,3,1
Can I build a settlement right next to another settlement?,Exception & corner case,67,12,8,1,7
Can I give away resource cards to another player as a gift?,Negation / constraint,37,7,5,1,4
Can I give away resource cards to another player as a gift?,Negation / constraint,79,14,7,2,5
How many points do I need to win?,Buried specific (ranks 4-10),39,7,4,1,3


## 10. Latency Profiling

Wall-clock time of the baseline retrieval versus the full re-ranking pipeline (baseline retrieval **plus** the cross-encoder pass). Retrieval-only timing is measured — the LLM call is excluded because it is identical for both pipelines. Each query is run several times and averaged.

In [11]:
def time_baseline(query, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        baseline_retrieve(query, TOP_K)
    return (time.perf_counter() - t0) / repeats


def time_rerank_pipeline(query, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        cand = baseline_retrieve(query, TOP_K)
        rerank(query, cand)
    return (time.perf_counter() - t0) / repeats


latency_rows = []
for item in FAILURE_QUERIES:
    q = item["query"]
    b = time_baseline(q)
    r = time_rerank_pipeline(q)
    latency_rows.append({"query": q,
                         "baseline_ms": round(b * 1000, 1),
                         "rerank_pipeline_ms": round(r * 1000, 1),
                         "overhead_ms": round((r - b) * 1000, 1),
                         "slowdown_x": round(r / b, 1)})

latency_df = pd.DataFrame(latency_rows)
(latency_df.style.hide(axis="index")
    .bar(subset=["rerank_pipeline_ms"], color="#f1aeb5")
    .bar(subset=["overhead_ms"], color="#ffe69c")
    .format({"baseline_ms": "{:.1f}", "rerank_pipeline_ms": "{:.1f}",
             "overhead_ms": "{:.1f}", "slowdown_x": "{:.1f}×"})
    .set_caption("Retrieval latency: baseline vs. full re-ranking pipeline (ms)"))

query,baseline_ms,rerank_pipeline_ms,overhead_ms,slowdown_x
What happens when a seven is rolled?,20.6,649.6,629.0,31.5×
Can I build a settlement right next to another settlement?,19.8,553.5,533.7,27.9×
Can I give away resource cards to another player as a gift?,21.1,420.9,399.8,19.9×
How many points do I need to win?,20.8,437.3,416.5,21.1×


## 11. Discussion pointers

The written analysis — architecture comparison, qualitative answer comparison, latency reading, and the scaling question — is in `report/REPORT.md`, backed by the tables produced above.

In short: the cross-encoder is accurate but costs a forward pass per candidate, so it is used only to re-rank a short baseline shortlist, never to score the whole corpus. Running it over every chunk for every query would multiply cost by the corpus size with no index to prune the search.